# Intermediate 05 lab — Grounded video understanding for an industrial workcell

This lab builds a timestamp-aware, inspectable video-language pipeline over deterministic procedural observations. It keeps the critical boundaries explicit:

> clock/source contract → sampling → temporal representation → retrieval → localization → deterministic reasoning → timestamped citation verification

The default `local_temporal_representation_proxy` and `local_video_language_proxy` are not production video encoders, VLMs, decoders, or quality benchmarks.

![A governed video-language pipeline from timestamped input to verified temporal claims.](assets/video-language-pipeline.svg)


## 1. Scenario, source policy, and safety boundary

- **Camera A / construction:** design the simulator, features, assertions, and failures.
- **Camera B / development only:** select sampling, segmentation, retrieval depth, and review policy.
- **Camera C / reporting only:** report once after policy freeze; no tuning.
- Ground-truth event annotations are stored separately from public frame observations.
- Sampling, feature, retrieval, and reranking functions accept no gold event IDs, boundaries, answers, or semantic labels.
- Video understanding is advisory: `authorization = "none"`.


In [ ]:
from __future__ import annotations

import hashlib
import inspect
import json
import math
import os
import platform
import random
import re
import sys
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field, fields, replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

SEED = 20260909
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

SOURCE_POLICY = {
    "Camera A": "construction",
    "Camera B": "development_only",
    "Camera C": "reporting_only_no_changes",
}
LOCAL_TEMPORAL_ENGINE = "local_temporal_representation_proxy"
LOCAL_VIDEO_LANGUAGE_ENGINE = "local_video_language_proxy"
DEMONSTRATION_THRESHOLD_NOTICE = "Demonstration thresholds for this notebook runtime only."

print({
    "seed": SEED,
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pillow": Image.__version__,
    "matplotlib": matplotlib.__version__,
    "source_policy": SOURCE_POLICY,
    "temporal_engine": LOCAL_TEMPORAL_ENGINE,
    "video_language_engine": LOCAL_VIDEO_LANGUAGE_ENGINE,
})


## 2. Typed video, frame, event, query, evidence, and claim contracts

Frame identity and timestamp coexist. Canonical temporal evidence is an interval. `VideoRequest` contains only inference-visible fields; evaluation annotations remain on `VideoQuery` and are stripped before any retrieval feature is built.


In [ ]:
@dataclass(frozen=True)
class Principal:
    principal_id: str
    tenant_id: str
    access_groups: tuple[str, ...]


@dataclass(frozen=True)
class VideoContract:
    video_id: str
    camera_id: str
    duration_seconds: float
    nominal_fps: float
    frame_count: int
    width: int
    height: int
    time_base: str
    clock_source: str
    source_split: str
    source_version: str
    brightness: float
    background_code: float
    variable_frame_rate: bool


@dataclass(frozen=True)
class FrameObservation:
    video_id: str
    camera_id: str
    frame_index: int
    timestamp_seconds: float
    operator_visible: float
    operator_x: float
    container_visible: float
    container_x: float
    valve_open_fraction: float
    pressure_norm: float
    leak_signal: float
    alarm_signal: float
    component_replaced: float
    motion_score: float
    audio_alarm_signal: float
    brightness: float
    background_code: float
    visible_track_ids: tuple[str, ...]


@dataclass(frozen=True)
class EventAnnotation:
    event_id: str
    video_id: str
    event_type: str
    start_seconds: float
    end_seconds: float
    track_ids: tuple[str, ...]
    source_split: str


@dataclass(frozen=True)
class ClipUnit:
    evidence_id: str
    video_id: str
    camera_id: str
    start_seconds: float
    end_seconds: float
    frame_indices: tuple[int, ...]
    timestamps: tuple[float, ...]
    features: tuple[float, ...]
    segmenter: str
    tenant_id: str
    access_groups: tuple[str, ...]
    source_version: str
    representation_version: str = "local-temporal-proxy-v1"
    index_version: str = "temporal-index-v1"


@dataclass(frozen=True)
class VideoRequest:
    query_id: str
    text: str
    principal: Principal
    source_split: str
    query_type: str
    target_event_types: tuple[str, ...]
    camera_filter: str


@dataclass(frozen=True)
class VideoQuery:
    query_id: str
    text: str
    principal: Principal
    source_split: str
    query_type: str
    target_event_types: tuple[str, ...]
    camera_filter: str
    required_event_ids: tuple[str, ...]
    gold_answer: Any = None


@dataclass(frozen=True)
class RetrievalHit:
    evidence_id: str
    score: float
    rank: int


@dataclass(frozen=True)
class TemporalEvidence:
    evidence_id: str
    video_id: str
    start_seconds: float
    end_seconds: float
    frame_start: int
    frame_end: int
    event_type: str
    track_ids: tuple[str, ...]
    source_version: str
    access_groups: tuple[str, ...]


@dataclass(frozen=True)
class TemporalClaim:
    claim_id: str
    text: str
    relation: str | None
    citation_ids: tuple[str, ...]
    authorization: str = "none"


@dataclass(frozen=True)
class StreamingState:
    event_type: str
    state: str
    first_seen_seconds: float | None
    completed_seconds: float | None
    last_observed_seconds: float


EVALUATION_ONLY_FIELDS = {
    "required_event_ids", "gold_answer", "event_annotations", "gold_start_seconds",
    "gold_end_seconds", "evaluation_only_semantic_class", "relevance_label",
}


def public_request(query: VideoQuery) -> VideoRequest:
    return VideoRequest(
        query_id=query.query_id,
        text=query.text,
        principal=query.principal,
        source_split=query.source_split,
        query_type=query.query_type,
        target_event_types=query.target_event_types,
        camera_filter=query.camera_filter,
    )


def stable_hash(payload: str) -> str:
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


assert EVALUATION_ONLY_FIELDS.isdisjoint({item.name for item in fields(VideoRequest)})
assert len(stable_hash("temporal evidence")) == 64


## 3. Procedural workcell and separately stored evaluation truth

The generator uses one private schedule to render public observations. After generation, only timestamped observable state is passed to the pipeline; event IDs and boundaries remain in a separate evaluation collection.


In [ ]:
CAMERA_CONFIG = {
    "Camera A": {"video_id": "workcell-a", "fps": 12.0, "duration": 28.0, "time_scale": .90, "brightness": 1.05, "background": .15, "vfr": False, "drop_every": None},
    "Camera B": {"video_id": "workcell-b", "fps": 10.0, "duration": 30.0, "time_scale": 1.00, "brightness": .90, "background": .45, "vfr": False, "drop_every": None},
    "Camera C": {"video_id": "workcell-c", "fps": 8.0, "duration": 34.0, "time_scale": 1.12, "brightness": .68, "background": .78, "vfr": True, "drop_every": 9},
}

BASE_EVENT_SCHEDULE = [
    ("operator_enter", 2.0, 3.0, ("operator_7",)),
    ("container_arrive", 4.5, 5.5, ("container_3",)),
    ("valve_open", 8.2, 8.6, ("valve_2",)),
    ("pressure_increase", 8.3, 11.0, ("valve_2",)),
    ("leak_begin", 12.0, 12.4, ("valve_2",)),
    ("alarm_activate", 14.0, 14.4, ("alarm_1",)),
    ("operator_close_valve", 16.0, 16.7, ("operator_7", "valve_2")),
    ("component_replace", 20.0, 23.0, ("operator_7", "connector_c17")),
    ("alarm_reset", 25.0, 25.4, ("operator_7", "alarm_1")),
]


def scaled_schedule(camera_id: str, overrides: dict[str, tuple[float, float]] | None = None) -> list[EventAnnotation]:
    config = CAMERA_CONFIG[camera_id]
    overrides = overrides or {}
    code = camera_id[-1].lower()
    events = []
    for index, (event_type, start, end, tracks) in enumerate(BASE_EVENT_SCHEDULE, 1):
        if event_type in overrides:
            start, end = overrides[event_type]
        else:
            start, end = start * config["time_scale"], end * config["time_scale"]
        events.append(EventAnnotation(
            event_id=f"{code}:event:{event_type}",
            video_id=config["video_id"],
            event_type=event_type,
            start_seconds=round(start, 4),
            end_seconds=round(end, 4),
            track_ids=tracks,
            source_split=SOURCE_POLICY[camera_id],
        ))
    return events


def transition_progress(timestamp: float, event: EventAnnotation) -> float:
    if timestamp <= event.start_seconds:
        return 0.0
    if timestamp >= event.end_seconds:
        return 1.0
    return (timestamp - event.start_seconds) / (event.end_seconds - event.start_seconds)


def event_map(events: list[EventAnnotation]) -> dict[str, EventAnnotation]:
    return {event.event_type: event for event in events}


def state_at(timestamp: float, events: list[EventAnnotation]) -> dict[str, float]:
    e = event_map(events)
    operator = transition_progress(timestamp, e["operator_enter"])
    container = transition_progress(timestamp, e["container_arrive"])
    valve_open = transition_progress(timestamp, e["valve_open"])
    valve_closed = transition_progress(timestamp, e["operator_close_valve"])
    valve = max(0.0, valve_open * (1.0 - valve_closed))
    pressure_up = transition_progress(timestamp, e["pressure_increase"])
    pressure = .35 + .55 * pressure_up * (1.0 - .8 * valve_closed)
    leak_on = transition_progress(timestamp, e["leak_begin"])
    leak = leak_on * (1.0 - valve_closed)
    alarm_on = transition_progress(timestamp, e["alarm_activate"])
    alarm_off = transition_progress(timestamp, e["alarm_reset"])
    alarm = alarm_on * (1.0 - alarm_off)
    replaced = transition_progress(timestamp, e["component_replace"])
    operator_x = -1.0 + 1.45 * operator + .25 * min(1.0, max(0.0, (timestamp - e["operator_close_valve"].start_seconds) / 4.0))
    container_x = -1.0 + 1.55 * container
    return {
        "operator_visible": operator,
        "operator_x": operator_x,
        "container_visible": container,
        "container_x": container_x,
        "valve_open_fraction": valve,
        "pressure_norm": pressure,
        "leak_signal": leak,
        "alarm_signal": alarm,
        "component_replaced": replaced,
    }


def build_camera(camera_id: str, overrides: dict[str, tuple[float, float]] | None = None):
    config = CAMERA_CONFIG[camera_id]
    events = scaled_schedule(camera_id, overrides)
    nominal_count = int(config["duration"] * config["fps"]) + 1
    nominal_times = np.arange(nominal_count, dtype=float) / config["fps"]
    if config["vfr"]:
        nominal_times = nominal_times + .018 * np.sin(np.arange(nominal_count) * .71)
        nominal_times = np.maximum.accumulate(nominal_times)
    frames, previous = [], None
    for original_index, timestamp in enumerate(nominal_times):
        if config["drop_every"] and original_index % config["drop_every"] == 4:
            continue
        state = state_at(float(timestamp), events)
        vector = np.array(list(state.values()), dtype=float)
        motion = float(np.abs(vector - previous).sum()) if previous is not None else 0.0
        previous = vector
        tracks = tuple(track for track, visible in (
            ("operator_7", state["operator_visible"]),
            ("container_3", state["container_visible"]),
            ("valve_2", 1.0),
            ("alarm_1", 1.0),
            ("connector_c17", 1.0),
        ) if visible > .05)
        frames.append(FrameObservation(
            video_id=config["video_id"], camera_id=camera_id, frame_index=original_index,
            timestamp_seconds=round(float(timestamp), 6), motion_score=motion,
            audio_alarm_signal=max(0.0, state["alarm_signal"] - (.15 if camera_id == "Camera C" else 0.0)),
            brightness=config["brightness"], background_code=config["background"],
            visible_track_ids=tracks, **state,
        ))
    contract = VideoContract(
        video_id=config["video_id"], camera_id=camera_id,
        duration_seconds=config["duration"], nominal_fps=config["fps"], frame_count=len(frames),
        width=640, height=360, time_base="1/90000", clock_source="synthetic_presentation_timestamp",
        source_split=SOURCE_POLICY[camera_id], source_version="1", brightness=config["brightness"],
        background_code=config["background"], variable_frame_rate=config["vfr"],
    )
    return contract, frames, events


video_contracts, public_frames, evaluation_events = {}, {}, {}
for camera in CAMERA_CONFIG:
    contract, frames_for_camera, events_for_camera = build_camera(camera)
    video_contracts[camera] = contract
    public_frames[camera] = frames_for_camera
    evaluation_events[camera] = events_for_camera

assert all(EVALUATION_ONLY_FIELDS.isdisjoint(asdict(frame)) for frames_for_camera in public_frames.values() for frame in frames_for_camera)
assert not math.isclose(
    public_frames["Camera C"][10].timestamp_seconds,
    public_frames["Camera C"][10].frame_index / video_contracts["Camera C"].nominal_fps,
)
pd.DataFrame([asdict(contract) for contract in video_contracts.values()])


## 4. Visualize public frames and the private evaluation timeline

These drawings are evidence illustrations, not inputs to a learned model. The timeline is available to the evaluator only; pipeline functions receive public observations.


In [ ]:
def draw_frame(frame: FrameObservation, size=(320, 180)) -> Image.Image:
    base = int(np.clip(210 * frame.brightness, 80, 235))
    image = Image.new("RGB", size, (base, base, min(250, base + 12)))
    draw = ImageDraw.Draw(image)
    draw.rectangle((40, 88, 280, 132), fill=(92, 105, 116), outline=(35, 47, 58), width=2)
    valve_color = (48, 170, 105) if frame.valve_open_fraction > .5 else (68, 92, 112)
    draw.ellipse((146, 91, 178, 123), fill=valve_color, outline=(26, 44, 58), width=2)
    if frame.operator_visible > .1:
        x = int(80 + 150 * np.clip(frame.operator_x, 0, 1))
        draw.ellipse((x, 48, x + 18, 66), fill=(37, 86, 151))
        draw.line((x + 9, 66, x + 9, 105), fill=(37, 86, 151), width=5)
    if frame.container_visible > .1:
        x = int(60 + 170 * np.clip(frame.container_x, 0, 1))
        draw.rectangle((x, 118, x + 28, 145), fill=(191, 133, 44), outline=(88, 64, 29))
    if frame.leak_signal > .15:
        draw.arc((140, 116, 188, 164), 15, 165, fill=(43, 132, 194), width=4)
    alarm_color = (222, 58, 46) if frame.alarm_signal > .5 else (80, 97, 106)
    draw.rectangle((250, 34, 276, 60), fill=alarm_color, outline=(40, 44, 48))
    draw.text((10, 10), f"{frame.camera_id} · t={frame.timestamp_seconds:.2f}s · i={frame.frame_index}", fill=(18, 42, 66))
    return image


camera_b = public_frames["Camera B"]
display(*[draw_frame(min(camera_b, key=lambda f: abs(f.timestamp_seconds - t))) for t in (1, 2.5, 8.4, 12.2, 14.2, 21.5)])

timeline = pd.DataFrame([asdict(event) for event in evaluation_events["Camera B"]])
fig, ax = plt.subplots(figsize=(10, 4.8))
for index, event in timeline.iterrows():
    ax.barh(event["event_type"], event["end_seconds"] - event["start_seconds"], left=event["start_seconds"], color="#2F6BFF")
ax.set_xlabel("timestamp (seconds)")
ax.set_title("Camera B evaluation timeline (not model input)")
plt.tight_layout()
timeline


## 5. Sampling by presentation time

Uniform sampling requests timestamps, then selects the nearest observed presentation time under a bounded tolerance. It does not reconstruct time from frame index.


In [ ]:
def uniform_sample(frames_for_video: list[FrameObservation], target_fps: float) -> list[FrameObservation]:
    if target_fps <= 0:
        raise ValueError("target_fps must be positive")
    end = frames_for_video[-1].timestamp_seconds
    targets = np.arange(0.0, end + 1e-9, 1.0 / target_fps)
    selected = []
    for target in targets:
        nearest = min(frames_for_video, key=lambda frame: abs(frame.timestamp_seconds - target))
        if nearest.frame_index not in {frame.frame_index for frame in selected}:
            selected.append(nearest)
    return selected


def event_aware_sample(frames_for_video: list[FrameObservation], base_fps=1.0, motion_threshold=.12) -> list[FrameObservation]:
    selected = {frame.frame_index: frame for frame in uniform_sample(frames_for_video, base_fps)}
    for index, frame in enumerate(frames_for_video):
        if frame.motion_score >= motion_threshold:
            for neighbor in range(max(0, index - 1), min(len(frames_for_video), index + 2)):
                selected[frames_for_video[neighbor].frame_index] = frames_for_video[neighbor]
    return sorted(selected.values(), key=lambda frame: frame.timestamp_seconds)


def random_sample(frames_for_video: list[FrameObservation], sample_count: int, seed=SEED) -> list[FrameObservation]:
    generator = np.random.default_rng(seed)
    indices = np.sort(generator.choice(len(frames_for_video), size=min(sample_count, len(frames_for_video)), replace=False))
    return [frames_for_video[int(index)] for index in indices]


def hierarchical_sample(frames_for_video: list[FrameObservation], coarse_fps=.5, motion_threshold=.12) -> list[FrameObservation]:
    """Coarse global coverage plus dense observations around public motion signals."""
    return event_aware_sample(frames_for_video, base_fps=coarse_fps, motion_threshold=motion_threshold)


def event_observation_recall(sampled: list[FrameObservation], annotations: list[EventAnnotation]) -> dict:
    observed = {
        event.event_id: any(event.start_seconds <= frame.timestamp_seconds <= event.end_seconds for frame in sampled)
        for event in annotations
    }
    return {"event_recall": float(np.mean(list(observed.values()))), "observed": observed}


sampling_rows = []
for camera in ("Camera B", "Camera C"):
    frames_for_camera = public_frames[camera]
    for label, sampled in [
        ("uniform_1_fps", uniform_sample(frames_for_camera, 1)),
        ("uniform_2_fps", uniform_sample(frames_for_camera, 2)),
        ("uniform_5_fps", uniform_sample(frames_for_camera, 5)),
        ("random_same_budget_as_1_fps", random_sample(frames_for_camera, len(uniform_sample(frames_for_camera, 1)))),
        ("event_aware", event_aware_sample(frames_for_camera)),
        ("hierarchical", hierarchical_sample(frames_for_camera)),
    ]:
        result = event_observation_recall(sampled, evaluation_events[camera])
        sampling_rows.append({"camera": camera, "policy": label, "frames_processed": len(sampled), "event_observation_recall": result["event_recall"]})

sampling_results = pd.DataFrame(sampling_rows)
assert len(uniform_sample(camera_b, 1)) < len(uniform_sample(camera_b, 5))
ax = sampling_results[sampling_results["camera"] == "Camera B"].plot.scatter(
    x="frames_processed", y="event_observation_recall", figsize=(7, 4), color="#2F6BFF"
)
for _, row in sampling_results[sampling_results["camera"] == "Camera B"].iterrows():
    ax.annotate(row["policy"], (row["frames_processed"], row["event_observation_recall"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_title("Sampling changes both cost and the observable event set")
plt.tight_layout()
sampling_results


## 6. Temporal aliasing: a sub-second event disappears

Camera B’s valve transition lasts 400 ms. The assertion below makes the failure concrete: integer-second samples miss it, while 5 FPS and event-aware sampling observe it.


In [ ]:
valve_event_b = next(event for event in evaluation_events["Camera B"] if event.event_type == "valve_open")


def event_is_observed(event: EventAnnotation, sampled: list[FrameObservation]) -> bool:
    return any(event.start_seconds <= frame.timestamp_seconds <= event.end_seconds for frame in sampled)


aliasing_example = {
    "event_interval": [valve_event_b.start_seconds, valve_event_b.end_seconds],
    "uniform_1_fps_observed": event_is_observed(valve_event_b, uniform_sample(camera_b, 1)),
    "uniform_5_fps_observed": event_is_observed(valve_event_b, uniform_sample(camera_b, 5)),
    "event_aware_observed": event_is_observed(valve_event_b, event_aware_sample(camera_b)),
}
assert aliasing_example["uniform_1_fps_observed"] is False
assert aliasing_example["uniform_5_fps_observed"] is True
assert aliasing_example["event_aware_observed"] is True
aliasing_example


## 7. Bag-of-frames versus order-aware representation

The controlled pair contains identical states in opposite order. Mean pooling must be identical. A delta feature must flip sign.


In [ ]:
def bag_of_frames_representation(sequence: np.ndarray) -> np.ndarray:
    return np.asarray(sequence, dtype=float).mean(axis=0)


def order_aware_representation(sequence: np.ndarray) -> np.ndarray:
    values = np.asarray(sequence, dtype=float)
    deltas = np.diff(values, axis=0)
    return np.concatenate([values.mean(axis=0), values[-1] - values[0], np.abs(deltas).sum(axis=0)])


closed_to_open = np.array([[0.0], [.25], [.75], [1.0]])
open_to_closed = closed_to_open[::-1]
bag_forward = bag_of_frames_representation(closed_to_open)
bag_reverse = bag_of_frames_representation(open_to_closed)
temporal_forward = order_aware_representation(closed_to_open)
temporal_reverse = order_aware_representation(open_to_closed)
assert np.allclose(bag_forward, bag_reverse)
assert not np.allclose(temporal_forward, temporal_reverse)
pd.DataFrame([
    {"sequence": "closed→open", "bag_mean": bag_forward[0], "temporal_delta": temporal_forward[1], "motion": temporal_forward[2]},
    {"sequence": "open→closed", "bag_mean": bag_reverse[0], "temporal_delta": temporal_reverse[1], "motion": temporal_reverse[2]},
])


## 8. Reverse and frame-shuffle dependence tests

These diagnostics ask whether an order-dependent classifier actually changes when order changes. `temporal_dependence_teaching` is a course diagnostic, not a standard benchmark metric.


In [ ]:
def direction_prediction(sequence: np.ndarray) -> str:
    delta = order_aware_representation(sequence)[1]
    return "opens" if delta > .05 else "closes" if delta < -.05 else "uncertain"


ordered_examples = [(closed_to_open, "opens"), (open_to_closed, "closes")]
rng = np.random.default_rng(SEED)
shuffled_examples = [(sequence[rng.permutation(len(sequence))], label) for sequence, label in ordered_examples]
ordered_accuracy = np.mean([direction_prediction(sequence) == label for sequence, label in ordered_examples])
shuffled_accuracy = np.mean([direction_prediction(sequence) == label for sequence, label in shuffled_examples])
temporal_dependence_teaching = float(ordered_accuracy - shuffled_accuracy)
reverse_video_test = {
    "forward": direction_prediction(closed_to_open),
    "reversed": direction_prediction(open_to_closed),
    "bag_representation_equal": bool(np.allclose(bag_forward, bag_reverse)),
}
assert reverse_video_test["forward"] != reverse_video_test["reversed"]
{"ordered_accuracy": ordered_accuracy, "shuffled_accuracy": shuffled_accuracy, "temporal_dependence_teaching": temporal_dependence_teaching, "reverse_video_test": reverse_video_test}


## 9. Clip features built only from public observations

The transparent representation captures observable state means, first-to-last deltas, and motion. No function receives event annotations or required event IDs.


In [ ]:
FEATURE_NAMES = (
    "operator_entry", "container_arrival", "valve_open", "valve_close",
    "leak", "alarm_activate", "component_replace", "alarm_reset", "motion",
)


def clip_features(frames_for_clip: list[FrameObservation]) -> np.ndarray:
    if not frames_for_clip:
        return np.zeros(len(FEATURE_NAMES), dtype=float)
    columns = np.array([[
        frame.operator_visible, frame.container_visible, frame.valve_open_fraction,
        frame.leak_signal, frame.alarm_signal, frame.component_replaced,
    ] for frame in frames_for_clip], dtype=float)
    delta = columns[-1] - columns[0]
    return np.array([
        max(delta[0], 0), max(delta[1], 0), max(delta[2], 0), max(-delta[2], 0),
        float(columns[:, 3].max()), max(delta[4], 0), max(delta[5], 0),
        max(-delta[4], 0), float(sum(frame.motion_score for frame in frames_for_clip)),
    ], dtype=float)


def frames_between(frames_for_video: list[FrameObservation], start: float, end: float) -> list[FrameObservation]:
    return [frame for frame in frames_for_video if start <= frame.timestamp_seconds <= end]


def make_clip(frames_for_video: list[FrameObservation], start: float, end: float, segmenter: str, ordinal: int) -> ClipUnit | None:
    selected = frames_between(frames_for_video, start, end)
    if len(selected) < 2:
        return None
    camera_id = selected[0].camera_id
    return ClipUnit(
        evidence_id=f"{selected[0].video_id}:{segmenter}:{ordinal:03d}",
        video_id=selected[0].video_id, camera_id=camera_id,
        start_seconds=selected[0].timestamp_seconds, end_seconds=selected[-1].timestamp_seconds,
        frame_indices=tuple(frame.frame_index for frame in selected),
        timestamps=tuple(frame.timestamp_seconds for frame in selected),
        features=tuple(clip_features(selected)), segmenter=segmenter,
        tenant_id="northstar", access_groups=("Operations", "Safety"), source_version="1",
    )


def fixed_windows(frames_for_video: list[FrameObservation], window_seconds=2.0, stride_seconds=1.0) -> list[ClipUnit]:
    end = frames_for_video[-1].timestamp_seconds
    clips = []
    for ordinal, start in enumerate(np.arange(0.0, max(0.0, end - window_seconds) + 1e-9, stride_seconds)):
        clip = make_clip(frames_for_video, float(start), float(start + window_seconds), "fixed_window", ordinal)
        if clip:
            clips.append(clip)
    return clips


def observable_change_segments(frames_for_video: list[FrameObservation], threshold=.12, radius_seconds=.7) -> list[ClipUnit]:
    candidate_times = [frame.timestamp_seconds for frame in frames_for_video if frame.motion_score >= threshold]
    centers = []
    for timestamp in candidate_times:
        if not centers or timestamp - centers[-1] > radius_seconds:
            centers.append(timestamp)
        else:
            centers[-1] = (centers[-1] + timestamp) / 2
    clips = []
    for ordinal, center in enumerate(centers):
        clip = make_clip(frames_for_video, max(0.0, center - radius_seconds), center + radius_seconds, "observable_change_point", ordinal)
        if clip:
            clips.append(clip)
    return clips


def oracle_event_segments(frames_for_video: list[FrameObservation], annotations: list[EventAnnotation]) -> list[ClipUnit]:
    """Evaluation-only ceiling. Never use this function as a deployable segmenter."""
    clips = []
    for ordinal, event in enumerate(annotations):
        clip = make_clip(frames_for_video, event.start_seconds, event.end_seconds, "oracle_event_segmentation_ceiling", ordinal)
        if clip:
            clips.append(clip)
    return clips


fixed_clip_indexes = {camera: fixed_windows(frames) for camera, frames in public_frames.items()}
change_clip_indexes = {camera: observable_change_segments(frames) for camera, frames in public_frames.items()}
oracle_clip_indexes = {camera: oracle_event_segments(public_frames[camera], evaluation_events[camera]) for camera in public_frames}

representation_example = pd.DataFrame([
    {"feature": name, "value": value}
    for name, value in zip(FEATURE_NAMES, change_clip_indexes["Camera B"][2].features)
])
assert all(clip.segmenter == "oracle_event_segmentation_ceiling" for clip in oracle_clip_indexes["Camera B"])
representation_example


## 10. Query contracts and public/evaluation separation

The public request names concepts the user asks for. Required event IDs and gold answers are evaluation-only and are removed before retrieval.


In [ ]:
OPS = Principal("operations-reviewer", "northstar", ("Operations",))


def queries_for(camera_id: str, split: str) -> list[VideoQuery]:
    code = camera_id[-1].lower()
    prefix = f"{code}:event:"
    return [
        VideoQuery(f"{code}-operator", "When does the operator enter?", OPS, split, "temporal_localization", ("operator_enter",), camera_id, (prefix + "operator_enter",)),
        VideoQuery(f"{code}-container", "Find the clip where the container arrives.", OPS, split, "text_to_clip", ("container_arrive",), camera_id, (prefix + "container_arrive",)),
        VideoQuery(f"{code}-valve", "When does the valve open?", OPS, split, "state_change", ("valve_open",), camera_id, (prefix + "valve_open",)),
        VideoQuery(f"{code}-leak", "When does the leak begin?", OPS, split, "temporal_localization", ("leak_begin",), camera_id, (prefix + "leak_begin",)),
        VideoQuery(f"{code}-alarm", "When does the alarm activate?", OPS, split, "temporal_localization", ("alarm_activate",), camera_id, (prefix + "alarm_activate",)),
        VideoQuery(f"{code}-replace", "When is connector C17 replaced?", OPS, split, "track_binding", ("component_replace",), camera_id, (prefix + "component_replace",)),
        VideoQuery(f"{code}-order", "Did the leak begin before the alarm activated?", OPS, split, "multi_event", ("leak_begin", "alarm_activate"), camera_id, (prefix + "leak_begin", prefix + "alarm_activate"), True),
        VideoQuery(f"{code}-duration", "How long was the alarm active?", OPS, split, "duration", ("alarm_activate", "alarm_reset"), camera_id, (prefix + "alarm_activate", prefix + "alarm_reset")),
    ]


development_queries = queries_for("Camera B", "development_only")
held_out_queries = queries_for("Camera C", "reporting_only_no_changes")
public_query_records = [asdict(public_request(query)) for query in development_queries]
evaluation_query_records = [{
    "query_id": query.query_id,
    "required_event_ids": query.required_event_ids,
    "gold_answer": query.gold_answer,
    "evaluation_only_semantic_class": query.target_event_types,
} for query in development_queries]
assert all(EVALUATION_ONLY_FIELDS.isdisjoint(record) for record in public_query_records)
assert all(query.source_split == "development_only" for query in development_queries)
pd.DataFrame([{"query_id": query.query_id, "type": query.query_type, "target_types": query.target_event_types, "gold_count": len(query.required_event_ids)} for query in development_queries])


## 11. Text-to-clip retrieval

The query proxy maps observable words into the same declared feature space. Retrieval receives authorized clips from one source and cannot read gold boundaries.


In [ ]:
QUERY_ALIASES = {
    "operator": "operator_entry", "enters": "operator_entry", "enter": "operator_entry",
    "container": "container_arrival", "arrives": "container_arrival", "arrive": "container_arrival",
    "opens": "valve_open", "open": "valve_open", "closes": "valve_close", "close": "valve_close",
    "leak": "leak", "alarm": "alarm_activate", "activates": "alarm_activate", "activate": "alarm_activate",
    "replaced": "component_replace", "replace": "component_replace", "reset": "alarm_reset",
}


def text_query_features(text: str, target_event_types: tuple[str, ...]) -> np.ndarray:
    vector = np.zeros(len(FEATURE_NAMES), dtype=float)
    normalized_types = {
        "operator_enter": "operator_entry", "container_arrive": "container_arrival",
        "alarm_activate": "alarm_activate", "component_replace": "component_replace",
        "leak_begin": "leak", "valve_open": "valve_open", "alarm_reset": "alarm_reset",
    }
    for event_type in target_event_types:
        feature = normalized_types.get(event_type, event_type)
        if feature in FEATURE_NAMES:
            vector[FEATURE_NAMES.index(feature)] += 1.0
    for token in re.findall(r"[a-z0-9]+", text.lower()):
        feature = QUERY_ALIASES.get(token)
        if feature in FEATURE_NAMES:
            vector[FEATURE_NAMES.index(feature)] += .25
    norm = np.linalg.norm(vector)
    return vector / norm if norm else vector


def cosine(first: np.ndarray, second: np.ndarray) -> float:
    denominator = np.linalg.norm(first) * np.linalg.norm(second)
    return float(first @ second / denominator) if denominator else 0.0


def authorized_clips(request: VideoRequest, clips: list[ClipUnit]) -> list[ClipUnit]:
    return [
        clip for clip in clips
        if clip.camera_id == request.camera_filter
        and clip.tenant_id == request.principal.tenant_id
        and bool(set(clip.access_groups).intersection(request.principal.access_groups))
    ]


def retrieve_clips(request: VideoRequest, clips: list[ClipUnit], top_k=5) -> list[RetrievalHit]:
    if not isinstance(request, VideoRequest):
        raise TypeError("retrieve_clips accepts only the public VideoRequest")
    query_vector = text_query_features(request.text, request.target_event_types)
    eligible = authorized_clips(request, clips)
    scores = [(clip.evidence_id, cosine(query_vector, np.asarray(clip.features))) for clip in eligible]
    ordered = sorted(scores, key=lambda item: (-item[1], item[0]))[:top_k]
    return [RetrievalHit(evidence_id, score, rank) for rank, (evidence_id, score) in enumerate(ordered, 1)]


PROXY_FEATURE_BUILDERS = {
    "uniform_sample": uniform_sample,
    "random_sample": random_sample,
    "event_aware_sample": event_aware_sample,
    "hierarchical_sample": hierarchical_sample,
    "clip_features": clip_features,
    "observable_change_segments": observable_change_segments,
    "text_query_features": text_query_features,
    "authorized_clips": authorized_clips,
    "retrieve_clips": retrieve_clips,
}


def assert_proxy_gold_separation() -> pd.DataFrame:
    rows = []
    for name, function in PROXY_FEATURE_BUILDERS.items():
        parameters = set(inspect.signature(function).parameters)
        forbidden = sorted(parameters.intersection(EVALUATION_ONLY_FIELDS))
        assert not forbidden, f"{name} receives evaluation-only inputs: {forbidden}"
        rows.append({"feature_builder": name, "parameters": sorted(parameters), "forbidden_parameters": forbidden})
    return pd.DataFrame(rows)


proxy_feature_contract = assert_proxy_gold_separation()
retrieval_example = retrieve_clips(public_request(development_queries[3]), fixed_clip_indexes["Camera B"], top_k=5)
display(proxy_feature_contract)
pd.DataFrame([asdict(hit) for hit in retrieval_example])


## 12. Temporal IoU and localization metrics

The implementation defines half-open intervals. Boundary-touching intervals have zero overlap. Known-answer tests cover perfect, partial, contained, disjoint, and touching cases.


In [ ]:
def temporal_iou(prediction: tuple[float, float], ground_truth: tuple[float, float]) -> float:
    ps, pe = prediction
    gs, ge = ground_truth
    if pe < ps or ge < gs:
        raise ValueError("interval end must be greater than or equal to start")
    intersection = max(0.0, min(pe, ge) - max(ps, gs))
    union = max(pe, ge) - min(ps, gs)
    return intersection / union if union else float(ps == gs)


tiou_known_cases = pd.DataFrame([
    ("perfect", temporal_iou((1, 3), (1, 3)), 1.0),
    ("partial", temporal_iou((1, 3), (2, 4)), 1/3),
    ("contained", temporal_iou((2, 3), (1, 4)), 1/3),
    ("no_overlap", temporal_iou((1, 2), (3, 4)), 0.0),
    ("boundary_touching", temporal_iou((1, 2), (2, 3)), 0.0),
], columns=["case", "observed", "expected"])
assert np.allclose(tiou_known_cases["observed"], tiou_known_cases["expected"])
tiou_known_cases


## 13. Candidate retrieval metrics

Candidate Recall@K asks whether a retrieved clip overlaps each required event. MRR measures the first relevant clip. This is a candidate gate—not yet precise temporal grounding.


In [ ]:
def event_lookup(camera_id: str) -> dict[str, EventAnnotation]:
    return {event.event_id: event for event in evaluation_events[camera_id]}


def clip_lookup(clips: Iterable[ClipUnit]) -> dict[str, ClipUnit]:
    return {clip.evidence_id: clip for clip in clips}


def candidate_retrieval_metrics(query: VideoQuery, hits: list[RetrievalHit], clips: list[ClipUnit], overlap_threshold=.08) -> dict:
    annotations = event_lookup(query.camera_filter)
    clip_by_id = clip_lookup(clips)
    gold = [annotations[event_id] for event_id in query.required_event_ids]
    matched_gold = set()
    first_rank = None
    for hit in hits:
        clip = clip_by_id[hit.evidence_id]
        relevant = False
        for event in gold:
            overlap = temporal_iou((clip.start_seconds, clip.end_seconds), (event.start_seconds, event.end_seconds))
            if overlap >= overlap_threshold:
                matched_gold.add(event.event_id)
                relevant = True
        if relevant and first_rank is None:
            first_rank = hit.rank
    return {
        "event_recall_at_k": len(matched_gold) / len(gold),
        "complete_temporal_evidence_recall": float(len(matched_gold) == len(gold)),
        "mrr": 1 / first_rank if first_rank else 0.0,
    }


def evaluate_retriever(queries: list[VideoQuery], clips: list[ClipUnit], top_k=5) -> pd.DataFrame:
    rows = []
    for query in queries:
        hits = retrieve_clips(public_request(query), clips, top_k)
        rows.append({"query_id": query.query_id, "query_type": query.query_type, **candidate_retrieval_metrics(query, hits, clips)})
    return pd.DataFrame(rows)


fixed_retrieval_report = evaluate_retriever(development_queries, fixed_clip_indexes["Camera B"], top_k=5)
change_retrieval_report = evaluate_retriever(development_queries, change_clip_indexes["Camera B"], top_k=5)
pd.concat({"fixed": fixed_retrieval_report, "observable_change": change_retrieval_report}, names=["segmenter"])


## 14. Fixed windows, observable change points, and oracle ceiling

The comparison keeps the same retrieval proxy and queries. Oracle boundaries measure an upper-bound segmentation condition; they are never mixed into the deployable result.


In [ ]:
segmentation_comparison_rows = []
for label, clips in [
    ("fixed_2s_stride_1s", fixed_clip_indexes["Camera B"]),
    ("observable_change_point", change_clip_indexes["Camera B"]),
    ("oracle_event_segmentation_ceiling", oracle_clip_indexes["Camera B"]),
]:
    report = evaluate_retriever(development_queries, clips, top_k=5)
    segmentation_comparison_rows.append({
        "segmenter": label,
        "clip_count": len(clips),
        "mean_event_recall_at_5": report["event_recall_at_k"].mean(),
        "complete_temporal_evidence_at_5": report["complete_temporal_evidence_recall"].mean(),
        "mrr": report["mrr"].mean(),
        "deployment_eligible": label != "oracle_event_segmentation_ceiling",
    })
segmentation_comparison = pd.DataFrame(segmentation_comparison_rows)
assert not segmentation_comparison.loc[segmentation_comparison["segmenter"] == "oracle_event_segmentation_ceiling", "deployment_eligible"].item()
segmentation_comparison


## 15. Dense boundary refinement from public state transitions

Candidate clips are coarse. The localizer uses signal changes within the top clips to estimate an event interval. It never reads annotation boundaries.


In [ ]:
EVENT_SIGNAL = {
    "operator_enter": ("operator_visible", 1),
    "container_arrive": ("container_visible", 1),
    "valve_open": ("valve_open_fraction", 1),
    "operator_close_valve": ("valve_open_fraction", -1),
    "leak_begin": ("leak_signal", 1),
    "alarm_activate": ("alarm_signal", 1),
    "component_replace": ("component_replaced", 1),
    "alarm_reset": ("alarm_signal", -1),
}


def localize_from_observations(event_type: str, frames_for_video: list[FrameObservation], candidate_clips: list[ClipUnit] | None = None) -> tuple[float, float] | None:
    signal_name, direction = EVENT_SIGNAL[event_type]
    allowed = None
    if candidate_clips:
        allowed = [(clip.start_seconds, clip.end_seconds) for clip in candidate_clips]
    selected = [
        frame for frame in frames_for_video
        if allowed is None or any(start <= frame.timestamp_seconds <= end for start, end in allowed)
    ]
    changes = []
    for first, second in zip(selected, selected[1:]):
        delta = getattr(second, signal_name) - getattr(first, signal_name)
        if direction * delta > .04:
            changes.append((first.timestamp_seconds, second.timestamp_seconds, abs(delta)))
    if not changes:
        return None
    strongest = max(change[2] for change in changes)
    meaningful = [change for change in changes if change[2] >= max(.04, strongest * .35)]
    return min(change[0] for change in meaningful), max(change[1] for change in meaningful)


def grounding_metrics(prediction: tuple[float, float] | None, event: EventAnnotation) -> dict:
    if prediction is None:
        return {"tiou": 0.0, "start_error_seconds": np.nan, "end_error_seconds": np.nan, "duration_error_seconds": np.nan}
    ground = (event.start_seconds, event.end_seconds)
    return {
        "tiou": temporal_iou(prediction, ground),
        "start_error_seconds": abs(prediction[0] - ground[0]),
        "end_error_seconds": abs(prediction[1] - ground[1]),
        "duration_error_seconds": abs((prediction[1] - prediction[0]) - (ground[1] - ground[0])),
    }


def ground_query(query: VideoQuery, clips: list[ClipUnit], top_k=5) -> list[dict]:
    request = public_request(query)
    hits = retrieve_clips(request, clips, top_k)
    by_id = clip_lookup(clips)
    candidate_clips = [by_id[hit.evidence_id] for hit in hits]
    annotations = event_lookup(query.camera_filter)
    rows = []
    for event_type, event_id in zip(request.target_event_types, query.required_event_ids):
        prediction = localize_from_observations(event_type, public_frames[query.camera_filter], candidate_clips)
        rows.append({"query_id": query.query_id, "event_type": event_type, "prediction": prediction, **grounding_metrics(prediction, annotations[event_id])})
    return rows


grounding_rows = [row for query in development_queries for row in ground_query(query, fixed_clip_indexes["Camera B"], top_k=8)]
grounding_report = pd.DataFrame(grounding_rows)
for threshold in (.3, .5, .7):
    grounding_report[f"recall_at_tiou_{threshold}"] = (grounding_report["tiou"] >= threshold).astype(float)
grounding_report


## 16. Deterministic interval relations and duration tools

Language models may propose events, but interval arithmetic should answer interval questions. The relation tool has known-answer tests for before, after, overlap, contains, during, and meets.


In [ ]:
def interval_relation(first: tuple[float, float], second: tuple[float, float], tolerance=.05) -> str:
    a_start, a_end = first
    b_start, b_end = second
    if abs(a_end - b_start) <= tolerance:
        return "meets"
    if abs(b_end - a_start) <= tolerance:
        return "met_by"
    if a_end < b_start:
        return "before"
    if b_end < a_start:
        return "after"
    if a_start <= b_start and a_end >= b_end:
        return "contains"
    if b_start <= a_start and b_end >= a_end:
        return "during"
    return "overlaps"


def interval_duration(interval: tuple[float, float]) -> float:
    start, end = interval
    if end < start:
        raise ValueError("interval end precedes start")
    return end - start


def build_temporal_event_graph(bundle: list[TemporalEvidence]) -> dict:
    ordered = sorted(bundle, key=lambda item: (item.start_seconds, item.end_seconds, item.evidence_id))
    edges = []
    for first_index, first in enumerate(ordered):
        for second in ordered[first_index + 1:]:
            edges.append({
                "source": first.evidence_id,
                "target": second.evidence_id,
                "relation": interval_relation(
                    (first.start_seconds, first.end_seconds),
                    (second.start_seconds, second.end_seconds),
                ),
            })
    return {"nodes": [asdict(item) for item in ordered], "edges": edges}


relation_known_cases = pd.DataFrame([
    ("before", interval_relation((1, 2), (3, 4)), "before"),
    ("after", interval_relation((5, 6), (3, 4)), "after"),
    ("overlaps", interval_relation((1, 3), (2, 4)), "overlaps"),
    ("contains", interval_relation((1, 5), (2, 4)), "contains"),
    ("during", interval_relation((2, 4), (1, 5)), "during"),
    ("meets", interval_relation((1, 2), (2, 3)), "meets"),
], columns=["case", "observed", "expected"])
assert (relation_known_cases["observed"] == relation_known_cases["expected"]).all()
relation_known_cases


## 17. Build canonical temporal evidence

Each evidence object binds an event hypothesis to a video, time interval, frame span, visible track IDs, source version, and access groups. Evidence IDs hash the canonical provenance fields rather than generated prose.


In [ ]:
PUBLIC_TRACK_HINTS = {
    "operator_enter": ("operator_7",),
    "container_arrive": ("container_3",),
    "valve_open": ("valve_2",),
    "operator_close_valve": ("operator_7", "valve_2"),
    "leak_begin": ("valve_2",),
    "alarm_activate": ("alarm_1",),
    "component_replace": ("operator_7", "connector_c17"),
    "alarm_reset": ("operator_7", "alarm_1"),
}


def nearest_frame(frames_for_video: list[FrameObservation], timestamp: float) -> FrameObservation:
    return min(frames_for_video, key=lambda frame: abs(frame.timestamp_seconds - timestamp))


def evidence_from_interval(camera_id: str, event_type: str, interval: tuple[float, float]) -> TemporalEvidence:
    frames_for_video = public_frames[camera_id]
    contract = video_contracts[camera_id]
    start_frame = nearest_frame(frames_for_video, interval[0])
    end_frame = nearest_frame(frames_for_video, interval[1])
    track_ids = tuple(
        track for track in PUBLIC_TRACK_HINTS[event_type]
        if any(track in frame.visible_track_ids for frame in frames_between(frames_for_video, interval[0], interval[1]))
    )
    identity = f"{contract.video_id}|{interval[0]:.4f}|{interval[1]:.4f}|{event_type}|{contract.source_version}"
    return TemporalEvidence(
        evidence_id="temporal:" + stable_hash(identity)[:16],
        video_id=contract.video_id,
        start_seconds=round(interval[0], 4), end_seconds=round(interval[1], 4),
        frame_start=start_frame.frame_index, frame_end=end_frame.frame_index,
        event_type=event_type, track_ids=track_ids, source_version=contract.source_version,
        access_groups=("Operations", "Safety"),
    )


def build_evidence_bundle(query: VideoQuery) -> list[TemporalEvidence]:
    bundle = []
    for event_type in query.target_event_types:
        interval = localize_from_observations(event_type, public_frames[query.camera_filter])
        if interval is not None:
            bundle.append(evidence_from_interval(query.camera_filter, event_type, interval))
    return bundle


example_bundle = build_evidence_bundle(development_queries[6])
example_event_graph = build_temporal_event_graph(example_bundle)
assert len(example_bundle) == 2
assert all(item.video_id == "workcell-b" for item in example_bundle)
assert example_event_graph["edges"][0]["relation"] == "before"
pd.DataFrame([asdict(item) for item in example_bundle])


## 18. Deduplicate overlapping event hypotheses

Sliding windows can emit repeated hypotheses for one event. Deduplication is scoped to the same video, event type, and track binding; unrelated events must not suppress one another.


In [ ]:
def deduplicate_evidence(items: list[TemporalEvidence], overlap_threshold=.5) -> list[TemporalEvidence]:
    kept = []
    for item in sorted(items, key=lambda value: (value.video_id, value.event_type, value.start_seconds, value.evidence_id)):
        duplicate = any(
            item.video_id == existing.video_id
            and item.event_type == existing.event_type
            and item.track_ids == existing.track_ids
            and temporal_iou(
                (item.start_seconds, item.end_seconds),
                (existing.start_seconds, existing.end_seconds),
            ) >= overlap_threshold
            for existing in kept
        )
        if not duplicate:
            kept.append(item)
    return kept


base_alarm = build_evidence_bundle(development_queries[4])[0]
duplicate_alarm = replace(
    base_alarm,
    evidence_id=base_alarm.evidence_id + ":duplicate",
    start_seconds=base_alarm.start_seconds + .02,
    end_seconds=base_alarm.end_seconds + .02,
)
unrelated_leak = build_evidence_bundle(development_queries[3])[0]
deduped = deduplicate_evidence([base_alarm, duplicate_alarm, unrelated_leak])
assert len(deduped) == 2
pd.DataFrame([asdict(item) for item in deduped])


## 19. Complete temporal evidence and multi-event reasoning

A relation answer is supported only when every required event is present. “Leak before alarm” cannot be justified by an alarm clip alone, even if the answer happens to be correct.


In [ ]:
def evidence_by_type(bundle: list[TemporalEvidence]) -> dict[str, TemporalEvidence]:
    return {item.event_type: item for item in bundle}


def answer_temporal_query(query: VideoQuery, bundle: list[TemporalEvidence]) -> dict:
    by_type = evidence_by_type(bundle)
    missing = [event_type for event_type in query.target_event_types if event_type not in by_type]
    if missing:
        return {"answer": "insufficient_evidence", "relation": None, "missing_event_types": missing}
    first, second = (by_type[event_type] for event_type in query.target_event_types[:2])
    relation = interval_relation(
        (first.start_seconds, first.end_seconds),
        (second.start_seconds, second.end_seconds),
    )
    return {
        "answer": relation in {"before", "meets"},
        "relation": relation,
        "missing_event_types": [],
        "citation_ids": (first.evidence_id, second.evidence_id),
    }


order_query = development_queries[6]
complete_bundle = build_evidence_bundle(order_query)
complete_answer = answer_temporal_query(order_query, complete_bundle)
incomplete_answer = answer_temporal_query(order_query, complete_bundle[1:])
assert complete_answer["answer"] is True
assert incomplete_answer["answer"] == "insufficient_evidence"
pd.DataFrame([
    {"condition": "complete bundle", **complete_answer},
    {"condition": "pre-event evidence removed", **incomplete_answer},
])


## 20. Temporal citation contract and deterministic verifier

The generator may draft a sentence, but the verifier owns acceptance. It checks citation existence, access, same-video provenance, interval validity, event coverage, track binding, and the claimed relation.


In [ ]:
def verify_temporal_claim(
    claim: TemporalClaim,
    bundle: list[TemporalEvidence],
    principal: Principal,
    required_event_types: tuple[str, ...],
) -> dict:
    indexed = {item.evidence_id: item for item in bundle}
    reasons = []
    cited = []
    for citation_id in claim.citation_ids:
        item = indexed.get(citation_id)
        if item is None:
            reasons.append("citation_not_in_bundle")
            continue
        cited.append(item)
        if not set(item.access_groups).intersection(principal.access_groups):
            reasons.append("citation_not_authorized")
        if item.end_seconds < item.start_seconds:
            reasons.append("invalid_interval")
        expected_tracks = set(PUBLIC_TRACK_HINTS.get(item.event_type, ()))
        if expected_tracks and not expected_tracks.intersection(item.track_ids):
            reasons.append("track_binding_missing")
    cited_types = {item.event_type for item in cited}
    if not set(required_event_types).issubset(cited_types):
        reasons.append("incomplete_temporal_evidence")
    if len({item.video_id for item in cited}) > 1:
        reasons.append("cross_video_relation")
    if claim.relation and len(cited) >= 2:
        observed = interval_relation(
            (cited[0].start_seconds, cited[0].end_seconds),
            (cited[1].start_seconds, cited[1].end_seconds),
        )
        if observed != claim.relation:
            reasons.append("unsupported_temporal_relation")
    reasons = sorted(set(reasons))
    return {
        "claim_id": claim.claim_id,
        "accepted": not reasons,
        "reasons": reasons,
        "authorization": claim.authorization,
    }


valid_claim = TemporalClaim(
    "claim-valid",
    "The leak began before the alarm activated.",
    "before",
    tuple(item.evidence_id for item in complete_bundle),
)
valid_verification = verify_temporal_claim(valid_claim, complete_bundle, OPS, order_query.target_event_types)
assert valid_verification["accepted"]
valid_verification


## 21. Automated temporal hallucination taxonomy

Each fixture changes one evidence property. These are executable reviewer tests, not labels inferred from generated prose.


In [ ]:
UNAUTHORIZED = Principal("visitor", "northstar", ("Visitor",))
bad_track_bundle = [replace(complete_bundle[0], track_ids=()), complete_bundle[1]]

hallucination_fixtures = [
    ("valid", valid_claim, complete_bundle, OPS, ()),
    ("invented_citation", replace(valid_claim, claim_id="bad-id", citation_ids=("temporal:not-real",)), complete_bundle, OPS, ("citation_not_in_bundle", "incomplete_temporal_evidence")),
    ("missing_pre_event", replace(valid_claim, claim_id="missing-event", citation_ids=(complete_bundle[1].evidence_id,)), complete_bundle, OPS, ("incomplete_temporal_evidence",)),
    ("wrong_relation", replace(valid_claim, claim_id="wrong-relation", relation="after"), complete_bundle, OPS, ("unsupported_temporal_relation",)),
    ("unauthorized", replace(valid_claim, claim_id="unauthorized"), complete_bundle, UNAUTHORIZED, ("citation_not_authorized",)),
    ("wrong_track", replace(valid_claim, claim_id="wrong-track"), bad_track_bundle, OPS, ("track_binding_missing",)),
]

hallucination_rows = []
for name, claim, bundle, principal, expected_reasons in hallucination_fixtures:
    result = verify_temporal_claim(claim, bundle, principal, order_query.target_event_types)
    assert set(expected_reasons).issubset(result["reasons"])
    hallucination_rows.append({"fixture": name, **result})
hallucination_report = pd.DataFrame(hallucination_rows)
hallucination_report


## 22. Evidence ablation test

Correct temporal answers should degrade when causally necessary evidence is removed. The verifier rejects the post-only and pre-only bundles as incomplete; an unrelated clip cannot repair them.


In [ ]:
ablation_conditions = {
    "complete": complete_bundle,
    "pre_event_only": complete_bundle[:1],
    "post_event_only": complete_bundle[1:],
    "unrelated_only": build_evidence_bundle(development_queries[1]),
    "complete_plus_unrelated": complete_bundle + build_evidence_bundle(development_queries[1]),
}
ablation_rows = []
for condition, bundle in ablation_conditions.items():
    answer = answer_temporal_query(order_query, bundle)
    claim = replace(valid_claim, claim_id=f"ablation-{condition}", citation_ids=tuple(item.evidence_id for item in bundle))
    verification = verify_temporal_claim(claim, bundle, OPS, order_query.target_event_types)
    ablation_rows.append({
        "condition": condition,
        "bundle_size": len(bundle),
        "answer": answer["answer"],
        "accepted": verification["accepted"],
        "reasons": verification["reasons"],
    })
ablation_report = pd.DataFrame(ablation_rows)
assert ablation_report.set_index("condition").loc["complete", "accepted"]
assert not ablation_report.set_index("condition").loc["post_event_only", "accepted"]
ablation_report


## 23. Relevant and irrelevant temporal counterfactuals

Swapping leak and alarm timing must flip the relation answer. Moving an unrelated container event must not. This tests evidence dependence more directly than ordinary accuracy.


In [ ]:
def counterfactual_relation(overrides: dict[str, tuple[float, float]]) -> dict:
    _, frames_cf, _ = build_camera("Camera B", overrides)
    intervals = {
        event_type: localize_from_observations(event_type, frames_cf)
        for event_type in ("leak_begin", "alarm_activate")
    }
    relation = interval_relation(intervals["leak_begin"], intervals["alarm_activate"])
    return {"relation": relation, "answer": relation in {"before", "meets"}, "intervals": intervals}


baseline_counterfactual = counterfactual_relation({})
relevant_counterfactual = counterfactual_relation({
    "leak_begin": (15.0, 15.4),
    "alarm_activate": (12.0, 12.4),
})
irrelevant_counterfactual = counterfactual_relation({"container_arrive": (6.0, 7.0)})
assert baseline_counterfactual["answer"] is True
assert relevant_counterfactual["answer"] is False
assert irrelevant_counterfactual["answer"] == baseline_counterfactual["answer"]
pd.DataFrame([
    {"condition": "baseline", **baseline_counterfactual},
    {"condition": "relevant timing swap", **relevant_counterfactual},
    {"condition": "irrelevant container shift", **irrelevant_counterfactual},
])


## 24. Long-video hierarchy: coarse search, fine evidence

Uniformly sending every frame to an expensive reasoner scales poorly. This proxy compares all-frame processing with a coarse clip index followed by fine inspection of retrieved intervals. Token counts are teaching proxies, not provider billing estimates.


In [ ]:
def hierarchy_cost(query: VideoQuery, frames_for_video: list[FrameObservation], clips: list[ClipUnit], top_k: int) -> dict:
    hits = retrieve_clips(public_request(query), clips, top_k)
    selected_ids = {hit.evidence_id for hit in hits}
    selected_frames = {
        frame_index
        for clip in clips if clip.evidence_id in selected_ids
        for frame_index in clip.frame_indices
    }
    metrics = candidate_retrieval_metrics(query, hits, clips)
    return {
        "query_id": query.query_id,
        "all_frame_units": len(frames_for_video),
        "coarse_clip_units": len(clips),
        "fine_frame_units": len(selected_frames),
        "hierarchical_total_units": len(clips) + len(selected_frames),
        "fine_fraction_of_all_frames": len(selected_frames) / len(frames_for_video),
        **metrics,
    }


hierarchy_report = pd.DataFrame([
    hierarchy_cost(query, public_frames["Camera B"], fixed_clip_indexes["Camera B"], top_k=5)
    for query in development_queries
])
hierarchy_report


## 25. Causal streaming state machine

Offline localization can inspect both sides of an event. A live system cannot. The state machine reports `possible_onset`, then `active`, and only declares completion after a falling edge. Detection delay is measured against separately held evaluation truth.


In [ ]:
def stream_event_state(
    frames_for_video: list[FrameObservation],
    event_type="alarm_activate",
    onset_threshold=.5,
    completion_threshold=.2,
) -> tuple[list[StreamingState], list[dict]]:
    signal_name = EVENT_SIGNAL[event_type][0]
    state = "not_observed"
    first_seen = None
    completed = None
    history, transitions = [], []
    for frame in frames_for_video:
        signal = getattr(frame, signal_name)
        prior = state
        if state == "not_observed" and signal >= onset_threshold:
            state = "possible_onset"
            first_seen = frame.timestamp_seconds
        elif state == "possible_onset" and signal >= onset_threshold:
            state = "active"
        elif state == "active" and signal <= completion_threshold:
            state = "complete"
            completed = frame.timestamp_seconds
        history.append(StreamingState(event_type, state, first_seen, completed, frame.timestamp_seconds))
        if state != prior:
            transitions.append({"timestamp_seconds": frame.timestamp_seconds, "from": prior, "to": state})
    return history, transitions


stream_history, stream_transitions = stream_event_state(public_frames["Camera B"])
alarm_truth = event_lookup("Camera B")["b:event:alarm_activate"]
reset_truth = event_lookup("Camera B")["b:event:alarm_reset"]
stream_summary = {
    "onset_detection_seconds": stream_history[-1].first_seen_seconds,
    "onset_delay_seconds": stream_history[-1].first_seen_seconds - alarm_truth.start_seconds,
    "completion_detection_seconds": stream_history[-1].completed_seconds,
    "completion_delay_seconds": stream_history[-1].completed_seconds - reset_truth.end_seconds,
    "state_contract": "causal; completion is unavailable before a falling edge",
}
assert stream_summary["onset_detection_seconds"] < stream_summary["completion_detection_seconds"]
display(pd.DataFrame(stream_transitions))
stream_summary


## 26. Backpressure policies and event preservation

When processing is slower than ingestion, latency can grow without bound. `queue_all` preserves every frame but accumulates delay. `keep_latest` bounds delay but can erase short transitions. `event_aware_drop` preserves high-motion frames while shedding redundant state.


In [ ]:
def simulate_backpressure(
    frames_for_video: list[FrameObservation],
    processing_fps: float,
    policy: str,
    motion_threshold=.12,
) -> dict:
    next_available = 0.0
    processed, dropped, delays = [], 0, []
    service_seconds = 1.0 / processing_fps
    pending_latest = None
    for frame in frames_for_video:
        arrival = frame.timestamp_seconds
        if policy == "queue_all":
            start = max(arrival, next_available)
            processed.append(frame)
            delays.append(start - arrival)
            next_available = start + service_seconds
        elif arrival >= next_available:
            chosen = pending_latest if pending_latest is not None else frame
            processed.append(chosen)
            delays.append(max(0.0, arrival - chosen.timestamp_seconds))
            next_available = arrival + service_seconds
            pending_latest = None
        elif policy == "event_aware_drop" and frame.motion_score >= motion_threshold:
            processed.append(frame)
            delays.append(max(0.0, next_available - arrival))
            next_available += service_seconds
        else:
            pending_latest = frame
            dropped += 1
    observed = event_observation_recall(processed, evaluation_events[frames_for_video[0].camera_id])
    return {
        "policy": policy,
        "processed_frames": len(processed),
        "dropped_or_replaced_frames": dropped,
        "median_delay_seconds": float(np.median(delays)) if delays else 0.0,
        "p95_delay_seconds": float(np.quantile(delays, .95)) if delays else 0.0,
        "event_observation_recall": observed["event_recall"],
    }


backpressure_report = pd.DataFrame([
    simulate_backpressure(public_frames["Camera B"], 3.0, policy)
    for policy in ("queue_all", "keep_latest", "event_aware_drop")
])
assert backpressure_report.set_index("policy").loc["queue_all", "p95_delay_seconds"] > 0
backpressure_report


## 27. Frame-drop sensitivity

Dropped observations change boundary resolution. This experiment removes increasing fractions of Camera B frames, reruns the same localizer, and reports tIoU rather than assuming graceful degradation.


In [ ]:
def deterministic_drop(frames_for_video: list[FrameObservation], keep_every: int) -> list[FrameObservation]:
    return [frame for ordinal, frame in enumerate(frames_for_video) if ordinal % keep_every == 0]


drop_rows = []
for keep_every in (1, 2, 4, 8):
    thinned = deterministic_drop(public_frames["Camera B"], keep_every)
    prediction = localize_from_observations("valve_open", thinned)
    metrics = grounding_metrics(prediction, event_lookup("Camera B")["b:event:valve_open"])
    drop_rows.append({
        "keep_every_nth_frame": keep_every,
        "effective_fraction": len(thinned) / len(public_frames["Camera B"]),
        "prediction": prediction,
        **metrics,
    })
frame_drop_report = pd.DataFrame(drop_rows)
frame_drop_report


## 28. Audio–visual alignment and conflict

Audio is another clocked evidence stream, not a free confidence boost. The local proxy compares first threshold crossings and abstains when the streams disagree beyond the declared tolerance.


In [ ]:
def first_crossing(frames_for_video: list[FrameObservation], field_name: str, threshold=.5) -> float | None:
    for frame in frames_for_video:
        if getattr(frame, field_name) >= threshold:
            return frame.timestamp_seconds
    return None


def audio_visual_alarm_check(frames_for_video: list[FrameObservation], audio_offset_seconds=0.0, tolerance=.35) -> dict:
    visual = first_crossing(frames_for_video, "alarm_signal")
    audio = first_crossing(frames_for_video, "audio_alarm_signal")
    shifted_audio = None if audio is None else audio + audio_offset_seconds
    aligned = visual is not None and shifted_audio is not None and abs(visual - shifted_audio) <= tolerance
    return {
        "visual_onset_seconds": visual,
        "audio_onset_seconds": shifted_audio,
        "absolute_offset_seconds": None if visual is None or shifted_audio is None else abs(visual - shifted_audio),
        "decision": "corroborated" if aligned else "conflict_abstain",
    }


av_report = pd.DataFrame([
    {"condition": "aligned", **audio_visual_alarm_check(public_frames["Camera B"])},
    {"condition": "audio shifted +1.2s", **audio_visual_alarm_check(public_frames["Camera B"], 1.2)},
])
assert av_report.set_index("condition").loc["aligned", "decision"] == "corroborated"
assert av_report.set_index("condition").loc["audio shifted +1.2s", "decision"] == "conflict_abstain"
av_report


## 29. Freeze the policy on Camera B

This configuration is the only one carried to Camera C. The values are pedagogical defaults, not production service objectives.


In [ ]:
FROZEN_POLICY = {
    "sampling": {"method": "event_aware", "base_fps": 1.0, "motion_threshold": .12},
    "segmenter": {"method": "fixed_window", "window_seconds": 2.0, "stride_seconds": 1.0},
    "retrieval_top_k": 8,
    "candidate_overlap_threshold": .08,
    "grounding_tiou_threshold": .30,
    "audio_visual_tolerance_seconds": .35,
    "threshold_notice": DEMONSTRATION_THRESHOLD_NOTICE,
    "selected_on": "Camera B development_only",
    "authorization": "none",
}
POLICY_HASH = stable_hash(json.dumps(FROZEN_POLICY, sort_keys=True))
FROZEN_POLICY | {"policy_hash": POLICY_HASH}


## 30. One-shot held-out Camera C report

Camera C changes brightness, background, cadence, presentation timestamps, and frame drops. This cell only reports the frozen policy. Do not edit policy values after reading it.


In [ ]:
def evaluate_source(camera_id: str, queries: list[VideoQuery]) -> dict:
    clips = fixed_windows(
        public_frames[camera_id],
        FROZEN_POLICY["segmenter"]["window_seconds"],
        FROZEN_POLICY["segmenter"]["stride_seconds"],
    )
    retrieval = evaluate_retriever(queries, clips, top_k=FROZEN_POLICY["retrieval_top_k"])
    grounding = pd.DataFrame([
        row for query in queries
        for row in ground_query(query, clips, top_k=FROZEN_POLICY["retrieval_top_k"])
    ])
    relation_queries = [query for query in queries if query.query_type == "multi_event"]
    relation_correct = []
    verifier_acceptance = []
    for query in relation_queries:
        bundle = build_evidence_bundle(query)
        answer = answer_temporal_query(query, bundle)
        relation_correct.append(float(answer["answer"] == query.gold_answer))
        claim = TemporalClaim(
            f"claim-{query.query_id}", "The leak began before the alarm activated.",
            answer.get("relation"), tuple(item.evidence_id for item in bundle),
        )
        verifier_acceptance.append(float(verify_temporal_claim(claim, bundle, query.principal, query.target_event_types)["accepted"]))
    return {
        "camera": camera_id,
        "source_split": video_contracts[camera_id].source_split,
        "policy_hash": POLICY_HASH,
        "candidate_event_recall_at_8": float(retrieval["event_recall_at_k"].mean()),
        "complete_temporal_evidence_at_8": float(retrieval["complete_temporal_evidence_recall"].mean()),
        "mean_tiou": float(grounding["tiou"].mean()),
        "recall_at_tiou_0_3": float((grounding["tiou"] >= FROZEN_POLICY["grounding_tiou_threshold"]).mean()),
        "mean_boundary_error_seconds": float(pd.concat([grounding["start_error_seconds"], grounding["end_error_seconds"]]).mean()),
        "relation_accuracy": float(np.mean(relation_correct)),
        "verified_claim_rate": float(np.mean(verifier_acceptance)),
    }


development_report = evaluate_source("Camera B", development_queries)
held_out_report = evaluate_source("Camera C", held_out_queries)
source_report = pd.DataFrame([development_report, held_out_report])
assert held_out_report["source_split"] == "reporting_only_no_changes"
assert development_report["policy_hash"] == held_out_report["policy_hash"]
source_report


## 31. Query taxonomy and evidence requirements

Different questions require different evidence. Presence can be answered from one interval; order, duration, causality, and cross-video questions need multiple grounded intervals and deterministic operators.


In [ ]:
query_taxonomy = pd.DataFrame([
    ("presence", "Was an alarm visible?", 1, "existence", "single convincing frame may suffice"),
    ("localization", "When did the leak begin?", 1, "boundary", "needs onset neighborhood"),
    ("state change", "Did the valve open or close?", 1, "ordered transition", "frame order is necessary"),
    ("order", "Did leak precede alarm?", 2, "interval relation", "complete temporal evidence required"),
    ("duration", "How long was the alarm active?", 2, "duration", "onset and completion required"),
    ("track binding", "Which component was replaced?", 1, "identity + interval", "appearance alone is insufficient"),
    ("causal", "Did the leak cause the alarm?", 2, "causal model", "temporal order alone cannot establish causality"),
    ("cross-video", "Which camera saw the event first?", 2, "synchronized clocks", "clock uncertainty must be represented"),
], columns=["query_type", "example", "minimum_intervals", "deterministic_tool", "evidence_warning"])
query_taxonomy


## 32. Temporal index record and multi-scale retrieval

An index record must retain timestamps, provenance, permissions, representation version, and segmentation policy. Multi-scale retrieval searches a coarse level first, then refines only the selected neighborhood.


In [ ]:
def temporal_index_record(clip: ClipUnit) -> dict:
    return {
        "evidence_id": clip.evidence_id,
        "video_id": clip.video_id,
        "camera_id": clip.camera_id,
        "start_seconds": clip.start_seconds,
        "end_seconds": clip.end_seconds,
        "frame_start": clip.frame_indices[0],
        "frame_end": clip.frame_indices[-1],
        "segmenter": clip.segmenter,
        "tenant_id": clip.tenant_id,
        "access_groups": clip.access_groups,
        "source_version": clip.source_version,
        "representation_version": clip.representation_version,
        "index_version": clip.index_version,
        "content_hash": stable_hash(f"{clip.video_id}|{clip.start_seconds}|{clip.end_seconds}|{clip.source_version}"),
    }


def multi_scale_retrieve(query: VideoQuery, coarse_clips: list[ClipUnit], frames_for_video: list[FrameObservation], coarse_k=3) -> dict:
    coarse_hits = retrieve_clips(public_request(query), coarse_clips, coarse_k)
    lookup = clip_lookup(coarse_clips)
    fine_clips = []
    for ordinal, hit in enumerate(coarse_hits):
        coarse = lookup[hit.evidence_id]
        center = (coarse.start_seconds + coarse.end_seconds) / 2
        for offset in (-.5, 0.0, .5):
            fine = make_clip(frames_for_video, max(0.0, center + offset - .5), center + offset + .5, "fine_refinement", ordinal * 3 + int((offset + .5) * 2))
            if fine:
                fine_clips.append(fine)
    fine_hits = retrieve_clips(public_request(query), fine_clips, min(5, len(fine_clips)))
    return {
        "coarse_candidates": len(coarse_hits),
        "fine_candidates": len(fine_clips),
        "fine_hits": len(fine_hits),
        "fine_evidence_ids": tuple(hit.evidence_id for hit in fine_hits),
    }


index_example = temporal_index_record(fixed_clip_indexes["Camera B"][0])
multiscale_example = multi_scale_retrieve(development_queries[4], fixed_clip_indexes["Camera B"], public_frames["Camera B"])
assert index_example["representation_version"] == "local-temporal-proxy-v1"
{**index_example, "features_omitted_from_display": True, "multiscale": multiscale_example}


## 33. Evidence-bound hierarchical summary

A long-video summary should be a collection of atomic, cited claims. It must preserve uncertainty and abstain where evidence is incomplete; generated fluency is not verification.


In [ ]:
SUMMARY_TEMPLATES = {
    "operator_enter": "An operator entered the workcell.",
    "container_arrive": "A container arrived at the station.",
    "valve_open": "The valve opened.",
    "leak_begin": "A leak signal began.",
    "alarm_activate": "The alarm activated.",
    "operator_close_valve": "The operator closed the valve.",
    "component_replace": "Connector C17 was replaced.",
    "alarm_reset": "The alarm reset.",
}


def evidence_bound_summary(camera_id: str) -> list[dict]:
    items = []
    for event_type, sentence in SUMMARY_TEMPLATES.items():
        interval = localize_from_observations(event_type, public_frames[camera_id])
        if interval is None:
            items.append({"event_type": event_type, "claim": "abstain", "citation_ids": (), "verified": False})
            continue
        evidence = evidence_from_interval(camera_id, event_type, interval)
        claim = TemporalClaim(f"summary-{camera_id}-{event_type}", sentence, None, (evidence.evidence_id,))
        verification = verify_temporal_claim(claim, [evidence], OPS, (event_type,))
        items.append({
            "event_type": event_type,
            "claim": sentence,
            "interval": interval,
            "citation_ids": claim.citation_ids,
            "verified": verification["accepted"],
        })
    return items


summary_items = evidence_bound_summary("Camera B")
assert sum(item["verified"] for item in summary_items) >= 6
assert all(item["verified"] or item["claim"] == "abstain" for item in summary_items)
pd.DataFrame(summary_items)


## 34. Language-prior challenge

A plausible narrative is not video evidence. The deliberately unsupported statement below is rejected because its cited bundle has no evacuation event and cannot satisfy the declared requirement.


In [ ]:
language_prior_claim = TemporalClaim(
    "language-prior",
    "After the alarm, the operator evacuated the workcell.",
    "after",
    (complete_bundle[1].evidence_id,),
)
language_prior_result = verify_temporal_claim(
    language_prior_claim,
    complete_bundle,
    OPS,
    ("alarm_activate", "operator_evacuate"),
)
assert not language_prior_result["accepted"]
assert "incomplete_temporal_evidence" in language_prior_result["reasons"]
language_prior_result


## 35. Governed adapter manifests — disabled by default

The default lab has no network or credential requirement. These examples are reviewable integration manifests, not executed downloads. Pin both package versions and model revisions, inspect model cards and licenses, avoid `trust_remote_code=True`, and keep decoded media inside the approved data boundary.


In [ ]:
OPTIONAL_ADAPTERS = {
    "pe_av_joint_embedding": {
        "enabled": False,
        "model_id": "facebook/pe-av-large",
        "revision": "0d24878d4107d64bef49e53602fc34ce6f94f6d8",
        "interfaces": ("AutoModel", "AutoProcessor"),
        "modalities": ("audio", "video", "text"),
        "license": "Apache-2.0 per pinned model card; re-verify before use",
        "trust_remote_code": False,
    },
    "videomae_action_encoder": {
        "enabled": False,
        "model_id": "MCG-NJU/videomae-base-finetuned-kinetics",
        "revision": "488eb9a0565f257b32866000305c8178965eb9f6",
        "interfaces": ("VideoMAEImageProcessor", "VideoMAEForVideoClassification"),
        "license": "CC-BY-NC-4.0 per pinned model card; non-commercial restriction matters",
        "trust_remote_code": False,
    },
    "qwen3_vl_generation": {
        "enabled": False,
        "model_id": "Qwen/Qwen3-VL-2B-Instruct",
        "revision": "89644892e4d85e24eaac8bacfd4f463576704203",
        "interfaces": ("AutoProcessor", "AutoModelForImageTextToText"),
        "license": "Apache-2.0 per pinned model card; re-verify dependencies and acceptable use",
        "trust_remote_code": False,
    },
}
assert all(not adapter["enabled"] for adapter in OPTIONAL_ADAPTERS.values())
assert all(adapter["trust_remote_code"] is False for adapter in OPTIONAL_ADAPTERS.values())
pd.DataFrame([
    {"adapter": name, "model_id": value["model_id"], "revision": value["revision"][:12], "enabled": value["enabled"], "license_review": value["license"]}
    for name, value in OPTIONAL_ADAPTERS.items()
])


## 36. Safe observability and enterprise decision artifact

Operational telemetry records versions, timing policy, aggregate outcomes, and hashed identities. It excludes raw frames, prompts, secrets, signed URLs, and face or badge crops. Deployment choice depends on task, evidence quality, compute, license, privacy, and reuse horizon—not leaderboard rank alone.


In [ ]:
SAFE_TELEMETRY_ALLOWLIST = {
    "request_id", "principal_hash", "camera_id", "source_split", "policy_hash",
    "representation_version", "index_version", "sampling_policy", "clips_retrieved",
    "verified_claims", "abstained_claims", "frame_drop_fraction", "authorization",
}


def safe_observability_record(query: VideoQuery, bundle: list[TemporalEvidence], accepted: bool) -> dict:
    record = {
        "request_id": stable_hash(query.query_id)[:16],
        "principal_hash": stable_hash(query.principal.principal_id)[:16],
        "camera_id": query.camera_filter,
        "source_split": query.source_split,
        "policy_hash": POLICY_HASH,
        "representation_version": "local-temporal-proxy-v1",
        "index_version": "temporal-index-v1",
        "sampling_policy": FROZEN_POLICY["sampling"]["method"],
        "clips_retrieved": len(bundle),
        "verified_claims": int(accepted),
        "abstained_claims": int(not accepted),
        "frame_drop_fraction": 1 - len(public_frames[query.camera_filter]) / (video_contracts[query.camera_filter].duration_seconds * video_contracts[query.camera_filter].nominal_fps + 1),
        "authorization": "none",
    }
    assert set(record) <= SAFE_TELEMETRY_ALLOWLIST
    return record


decision_artifact = pd.DataFrame([
    ("Transparent local proxy", "contract/evaluation education", "none", "low", "high", "not a production model", "keep for tests"),
    ("VideoMAE adapter", "action representation", "CC-BY-NC-4.0 checkpoint", "medium", "self-hosted", "commercial restriction", "research evaluation only"),
    ("PE-AV adapter", "joint audio-video-text retrieval", "Apache-2.0 model card", "high", "self-hosted", "2B model operations", "pilot if multimodal retrieval is needed"),
    ("Qwen3-VL adapter", "video-conditioned generation", "Apache-2.0 model card", "high", "self-hosted/API dependent", "hallucination + token cost", "only behind evidence verifier"),
], columns=["option", "best_fit", "license_signal", "compute", "data_boundary", "primary_risk", "decision"])

telemetry_example = safe_observability_record(order_query, complete_bundle, valid_verification["accepted"])
display(pd.DataFrame([telemetry_example]))
decision_artifact


## 37. Save a governed evidence artifact

The artifact is machine-readable and explicitly labels the local proxies. It contains aggregate evidence and policy metadata—not raw video, hidden annotations, credentials, or autonomous actions.


In [ ]:
artifact = {
    "course": "Intermediate 05 — Video-Language Understanding",
    "created_at": "2026-09-09T00:00:00Z",
    "engine": LOCAL_VIDEO_LANGUAGE_ENGINE,
    "temporal_representation_engine": LOCAL_TEMPORAL_ENGINE,
    "foundation_model": False,
    "learned_quality_benchmark": False,
    "authorization": "none",
    "threshold_notice": DEMONSTRATION_THRESHOLD_NOTICE,
    "source_policy": SOURCE_POLICY,
    "policy": FROZEN_POLICY,
    "policy_hash": POLICY_HASH,
    "development_report": development_report,
    "held_out_report": held_out_report,
    "temporal_evidence": [asdict(item) for item in complete_bundle],
    "claim": asdict(valid_claim),
    "claim_verification": valid_verification,
    "hallucination_test_counts": {
        str(key): int(value) for key, value in hallucination_report["accepted"].value_counts().items()
    },
    "counterfactual_checks": {
        "baseline_answer": baseline_counterfactual["answer"],
        "relevant_swap_answer": relevant_counterfactual["answer"],
        "irrelevant_shift_answer": irrelevant_counterfactual["answer"],
    },
    "optional_adapters": OPTIONAL_ADAPTERS,
    "telemetry_example": telemetry_example,
}
artifact_path = Path(".artifacts/intermediate-05-video-language-evidence.json")
artifact_path.parent.mkdir(parents=True, exist_ok=True)
artifact_path.write_text(json.dumps(artifact, indent=2, sort_keys=True), encoding="utf-8")
assert json.loads(artifact_path.read_text(encoding="utf-8"))["foundation_model"] is False
print(f"Saved {artifact_path} ({artifact_path.stat().st_size:,} bytes)")


## 38. Production upgrade path

1. Replace procedural observations with approved decoding (for example TorchCodec) while preserving presentation timestamps and decode failures.
2. Replace the local representation proxy with a pinned, licensed video encoder; retain source-isolated evaluation and provenance.
3. Add a governed vector index with pre-query authorization and deletion propagation.
4. Use a video-language generator only behind typed evidence bundles and deterministic claim verification.
5. Benchmark decode, sampling, encoding, retrieval, localization, and generation separately on target hardware.
6. Add privacy review, retention, biometric restrictions, human escalation, incident response, and rollback before any live deployment.

Never treat this notebook's synthetic accuracy, compute units, or thresholds as a production service-level claim.


## 39. Exercises

1. Change `target_fps` and identify the first event missed. Explain why frame index cannot repair it on Camera C.
2. Add a query requiring three events. Show complete-evidence recall separately from per-event recall.
3. Create a cross-video query and extend the evidence contract with clock uncertainty.
4. Add a topology-preserving track handoff and test that deduplication does not merge different objects.
5. Shift audio by 200 ms, 500 ms, and 1 s. Choose an abstention policy before viewing held-out results.
6. Add a retention deadline to `ClipUnit` and prove expired evidence cannot be retrieved.
7. Compare fixed, observable-change, and oracle segment boundaries without presenting the oracle ceiling as deployable.
8. Replace one local proxy with a pinned adapter in an isolated environment and document model, data, license, latency, and failure evidence.


## 40. What you should now be able to explain without code

- Why identical frames in a different order can require a different answer.
- Why one FPS may erase an event rather than merely reduce visual quality.
- Why candidate retrieval, temporal grounding, and answer verification require separate metrics.
- Why a correct relation answer can still be unsupported when one event is missing.
- Why timestamps, track identity, access policy, and source version belong in a temporal citation.
- Why temporal order is necessary but not sufficient evidence for causality.
- Why streaming onset and offline completion are different product contracts.
- Why a fluent video summary is not trustworthy without atomic, verified temporal claims.
- Why a current foundation model does not remove the need for sampling, evaluation, governance, or deterministic tools.

**Next:** Intermediate 06 turns grounded multimodal evidence into carefully bounded visual-agent workflows.
